# 4. Embeddings and retrieval

**Learning objective:** understand how validated records become searchable
vectors, how the retriever keeps one loud document from crowding out the rest,
and how retrieval quality is measured rather than assumed.

**Where this fits:** this notebook covers the two steps between the knowledge
base and the generator.

```
SourceRecord list -> embed -> local Qdrant collection -> retrieve -> RetrievedRecord list
```

Everything here runs offline. The default path downloads no models and opens no
database.

## Embeddings as coordinates

An embedding model maps text to a point in a few hundred dimensions, placed so
that texts about the same thing land near each other. "A gentle weekly rhythm"
and "an unhurried routine for the week" share almost no words but end up close
together, which is exactly what keyword search cannot do.

Closeness is measured with cosine similarity: the dot product of two vectors that
have been scaled to unit length. Because the vectors are normalized, the number
is a bounded similarity rather than an unbounded distance, and it can be compared
across queries.

This project embeds locally with `sentence-transformers/all-MiniLM-L6-v2`, a
384-dimensional model that runs on a laptop CPU. Local embedding keeps the
private Mosaic text off the network, which is a privacy decision before it is a
cost decision.

In [ ]:
import math
from typing import Any

from mosaic_pathway.app_support import evidence_preview
from mosaic_pathway.embeddings import DEFAULT_EMBEDDING_MODEL, LocalEmbeddingModel
from mosaic_pathway.models import RetrievalExample, SourceRecord
from mosaic_pathway.retrieval import (
    DEFAULT_MAX_PER_SOURCE,
    DEFAULT_TOP_K,
    MosaicRetriever,
    candidate_limit,
    inventory_source_id,
    select_diverse_records,
)
from mosaic_pathway.retrieval_evaluation import evaluate_examples, summarize
from mosaic_pathway.vector_store import (
    COLLECTION_NAME,
    VECTOR_STORE_PATH,
    MosaicVectorStore,
    point_id_for,
)

print("embedding model:", DEFAULT_EMBEDDING_MODEL)
print("collection name:", COLLECTION_NAME)
print("defaults: top_k =", DEFAULT_TOP_K, "max_per_source =", DEFAULT_MAX_PER_SOURCE)

## A tiny stand-in embedding model

To keep this notebook offline and fast, the demonstrations below use a bag of
words over an eight-word vocabulary instead of the real model. The interface is
identical to `LocalEmbeddingModel`, so the production retriever cannot tell the
difference.

The stand-in has no semantic ability at all: it only counts words. That is the
point. It isolates the retrieval mechanics from the embedding quality, and the
real model is available further down behind a flag.

In [ ]:
VOCABULARY = [
    "animals",
    "drawing",
    "outdoors",
    "rhythm",
    "community",
    "reading",
    "teen",
    "criticism",
]


class BagOfWordsEmbeddingModel:
    """Deterministic offline stand-in with the same interface as the real model."""

    model_name = "bag-of-words-demo"

    def __init__(self, vocabulary: list[str]) -> None:
        self.vocabulary = vocabulary
        self.dimension = len(vocabulary)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._vector(text) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._vector(text)

    def _vector(self, text: str) -> list[float]:
        lowered = text.lower()
        raw = [float(lowered.count(word)) for word in self.vocabulary]
        length = math.sqrt(sum(value * value for value in raw)) or 1.0

        return [value / length for value in raw]


def cosine(left: list[float], right: list[float]) -> float:
    return sum(a * b for a, b in zip(left, right, strict=True))


embedding_model = BagOfWordsEmbeddingModel(VOCABULARY)

rhythm_only = embedding_model.embed_query("rhythm")
rhythm_and_animals = embedding_model.embed_query("rhythm and animals")
drawing_only = embedding_model.embed_query("drawing")

print("rhythm vs rhythm+animals:", round(cosine(rhythm_only, rhythm_and_animals), 3))
print("rhythm vs drawing       :", round(cosine(rhythm_only, drawing_only), 3))

## The vector store: points, vectors, and payloads

Qdrant stores points. Each point has an id, a vector, and a payload. In this
project the payload is the complete serialized `SourceRecord`, so a search
returns everything needed to build context without a second lookup.

Qdrant runs in local persistent mode: a directory on disk rather than a server.
That is the right size for this project and comes with one important constraint
covered in notebook 7 -- the directory is locked by a single process.

Point ids are UUIDs derived deterministically from the record id with `uuid5`.
Re-indexing the same record overwrites the same point instead of creating a
duplicate, which makes an index rebuild idempotent.

In [ ]:
print(point_id_for("synthetic-guide-0007"))
print(point_id_for("synthetic-guide-0007"), "<- same record id, same point id")
print(point_id_for("synthetic-guide-0008"), "<- different record id")

In [ ]:
class InMemoryVectorStore:
    """Offline stand-in for the Qdrant collection used by the real retriever."""

    def __init__(
        self, records: list[SourceRecord], model: BagOfWordsEmbeddingModel
    ) -> None:
        self.payloads = [record.model_dump() for record in records]
        self.vectors = model.embed_documents([record.text for record in records])
        self.requested_limits: list[int] = []

    def count(self) -> int:
        return len(self.payloads)

    def search(
        self, query_vector: list[float], limit: int
    ) -> list[tuple[dict[str, Any], float]]:
        self.requested_limits.append(limit)
        scored = [
            (payload, cosine(vector, query_vector))
            for payload, vector in zip(self.payloads, self.vectors, strict=True)
        ]
        scored.sort(key=lambda item: item[1], reverse=True)

        return scored[:limit]


def make_record(source: str, index: int, text: str) -> SourceRecord:
    return SourceRecord(
        source_id=f"{source}-{index:04d}",
        title=f"{source} chunk {index}",
        source_file=f"{source}.docx",
        content_type="practical_guidance",
        authority_type="mosaic_guidance",
        topics=["synthetic"],
        text=text,
    )


records = (
    [
        make_record("synthetic-busy", index, "A short note about rhythm for the week.")
        for index in range(1, 25)
    ]
    + [
        make_record(
            "synthetic-varied",
            index,
            "A passage about rhythm and animals in daily life.",
        )
        for index in range(1, 7)
    ]
    + [
        make_record(
            "synthetic-third",
            index,
            "A passage about rhythm and community for families.",
        )
        for index in range(1, 5)
    ]
)

vector_store = InMemoryVectorStore(records, embedding_model)

print("records indexed:", vector_store.count())
print("sources:", sorted({inventory_source_id(record) for record in records}))

## Why a per-source cap exists

The corpus is not balanced. One large document can supply the majority of the
chunks, and pure similarity ranking will happily return five chunks from that
one document. The family then gets a pathway that quotes a single source five
times and calls it evidence.

`select_diverse_records` caps how many chunks any one inventory source may
contribute, keeping the highest scoring ones within that cap.

In [ ]:
candidates = vector_store.search(embedding_model.embed_query("rhythm"), 20)

uncapped = select_diverse_records(candidates, top_k=5, max_per_source=5)
capped = select_diverse_records(candidates, top_k=5, max_per_source=2)

print("without a meaningful cap:", [item.record.source_id for item in uncapped])
print("with max_per_source = 2 :", [item.record.source_id for item in capped])

## Adaptive candidate expansion

The cap creates a second problem. If the first window of candidates is entirely
filled by the dominant source, capping it leaves too few results.

So the retriever over-fetches first, and then widens the window and searches
again until it can fill `top_k` or it has considered the whole collection.

In [ ]:
print("candidate window for top_k = 5:", candidate_limit(5))

retriever = MosaicRetriever(embedding_model, vector_store)

vector_store.requested_limits.clear()
capped_result = retriever.retrieve("rhythm", top_k=5, max_per_source=2)
print("capped run  ->", [item.record.source_id for item in capped_result.records])
print("windows tried:", vector_store.requested_limits)

vector_store.requested_limits.clear()
open_result = retriever.retrieve("rhythm", top_k=5, max_per_source=5)
print()
print("uncapped run ->", [item.record.source_id for item in open_result.records])
print("windows tried:", vector_store.requested_limits)

The capped run had to widen its window because the first 20 candidates were all
from the dominant source. The uncapped run was satisfied immediately.

This is the trade-off the retriever makes explicit: diversity costs extra search
work, and in a corpus this size that cost is trivial compared to returning five
chunks of the same document.

## Measuring retrieval instead of eyeballing it

Two metrics are enough for a project this size.

**Hit rate at k** asks a blunt question: for a query where we know which document
should answer it, did any chunk of that document appear in the top k results? It
does not care about position.

**Mean reciprocal rank** does care about position. A hit at rank 1 scores 1.0, at
rank 2 scores 0.5, at rank 3 scores 0.33. It rewards putting the right evidence
near the top, which matters because the generator sees the whole context but
weights the top of it more.

The evaluation set is a small file of queries with expected source ids. The
production functions below run against any retriever, including the offline one
built in this notebook.

In [ ]:
examples = [
    RetrievalExample(
        query_id="rhythm-general",
        query="rhythm",
        expected_source_ids=["synthetic-varied"],
    ),
    RetrievalExample(
        query_id="animal-interest",
        query="animals and drawing for a curious child",
        expected_source_ids=["synthetic-varied"],
    ),
    RetrievalExample(
        query_id="handling-criticism",
        query="how to handle criticism from extended family",
        expected_source_ids=["synthetic-support"],
    ),
]

evaluations = evaluate_examples(retriever, examples)

for evaluation in evaluations:
    outcome = f"hit at rank {evaluation.first_hit_rank}" if evaluation.hit else "miss"
    print(f"{evaluation.query_id:<20} {outcome}")
    print(f"{'':<20} retrieved: {evaluation.retrieved_source_ids}")

summary = summarize(evaluations)
print()
print("queries:", summary.queries_evaluated)
print("hits at k:", summary.hits_at_k)
print("hit rate:", round(summary.hit_rate_at_k, 3))
print("mean reciprocal rank:", round(summary.mean_reciprocal_rank, 3))

The third query is a deliberate miss. It asks for something the corpus does not
contain, and the retriever answers anyway with the closest thing it has. A
retriever cannot return nothing; a missing topic becomes a confidently wrong
result rather than an empty one.

Note also that the mean reciprocal rank here averages over hits only. That is a
reporting choice made in `summarize`, and it means MRR describes ranking quality
for queries that worked, not overall system quality.

## The recorded baseline for the real corpus

Running the same functions against the real index gave:

| Metric | Value |
| --- | --- |
| Queries evaluated | 10 |
| Hits at five | 8 |
| Hit rate at five | 0.80 |

The two misses were a query about handling criticism from family, and a query
about a child moving from an educator-led setting to a parent-led one. Both are
corpus gaps rather than ranking failures: the material simply does not cover
those topics in depth.

Ten hand-written queries is a smoke test, not a statistically rigorous
evaluation. It is enough to notice a regression and nowhere near enough to
compare two embedding models with confidence.

## Optional: the real local embedding model

The cell below downloads and runs the real sentence-transformer model the first
time it executes, so it is disabled by default. Enable it to see genuine semantic
similarity, which the bag-of-words stand-in cannot show.

In [ ]:
RUN_LOCAL_EMBEDDING_MODEL = False

if RUN_LOCAL_EMBEDDING_MODEL:
    local_model = LocalEmbeddingModel()
    vectors = local_model.embed_documents(
        [
            "A gentle weekly rhythm for a family.",
            "An unhurried routine for the week at home.",
            "Filing a quarterly tax return online.",
        ]
    )

    print("dimension:", local_model.dimension)
    print("paraphrase pair :", round(cosine(vectors[0], vectors[1]), 3))
    print("unrelated pair  :", round(cosine(vectors[0], vectors[2]), 3))
else:
    print("Skipped: set RUN_LOCAL_EMBEDDING_MODEL to True to load the real model.")

## Optional: query the real local index

This section needs a built vector store and the real embedding model, and it
reads private Mosaic text, so it is disabled by default and prints only short
truncated previews. Clear the notebook outputs after running it.

Close any running Streamlit app or API process first: local Qdrant allows one
process to hold the directory at a time.

In [ ]:
INSPECT_LOCAL_VECTOR_STORE = False

if not INSPECT_LOCAL_VECTOR_STORE:
    print("Skipped: set INSPECT_LOCAL_VECTOR_STORE to True to query the local index.")
elif not VECTOR_STORE_PATH.is_dir():
    print(
        "No local index found. Build one with: uv run python -m mosaic_pathway.vector_store"
    )
else:
    with MosaicVectorStore() as store:
        print("points in collection:", store.count())

        local_retriever = MosaicRetriever(LocalEmbeddingModel(), store)
        result = local_retriever.retrieve(
            "a family wanting more outdoor time and a gentler weekly rhythm",
            top_k=DEFAULT_TOP_K,
            max_per_source=DEFAULT_MAX_PER_SOURCE,
        )

        for rank, item in enumerate(result.records, start=1):
            print(
                f"{rank}. {item.score:.3f} {item.record.source_id} | {item.record.title}"
            )
            print(f"   {evidence_preview(item.record.text, limit=200)}")

## What vector-only retrieval cannot do

* It cannot say "I do not know". Every query returns the top k of something.
* It has no lexical fallback, so an exact term such as a book title can be missed
  when the surrounding language differs.
* It has no reranking stage, so a chunk that is broadly on-topic can outrank a
  chunk that answers the question directly.
* It has no query understanding: a long intake-derived query is one blended
  vector, and a specific need inside it can be averaged away.
* Similarity is not support. A retrieved chunk being close to the query does not
  mean it justifies a recommendation, which is the gap notebook 6 keeps a human
  review rubric for.

## Key takeaways

* Embeddings turn text into comparable coordinates; cosine similarity over
  normalized vectors is the whole ranking mechanism.
* Payloads carry the full record, so search results need no second lookup.
* Deterministic `uuid5` point ids make re-indexing idempotent.
* The per-source cap protects against corpus imbalance, and adaptive expansion
  pays for that cap by widening the search window when it needs to.
* Hit rate and MRR are cheap, useful, and small enough to be honest about.

## Next

Notebook 5 joins retrieval and generation into a single grounded service and adds
the first automatic check that a pathway only cites evidence it was actually
given.